- NAME          : ANISH SAMUEL
- COURSE        : DSC670-T301 Advanced Uses of Generative AI
- ASSIGNMENT    : WEEK 6

# Fine-Tuning an LLM with LoRA

**The goal:** teach a large language model (Mistral-7B) to follow instructions in a set format, without retraining the whole model.

**The idea behind LoRA:** instead of changing all 7 billion numbers in the model, we freeze them and add a few small "adapter" layers. Only the adapters get trained, which is fast and needs much less memory.

**Before running:** this needs an NVIDIA GPU. In Google Colab, set Runtime → Change runtime type → **T4 GPU**.

## Step 1: Install the libraries

The author uses five Hugging Face tools:
- `transformers` loads the model and trains it
- `peft` adds the LoRA adapters
- `accelerate` handles the GPU
- `datasets` downloads the training data
- `bitsandbytes` shrinks the model so it fits in memory

In [1]:
!pip install -q -U transformers peft accelerate datasets bitsandbytes

## Step 2: Load the model

A 7-billion-parameter model is normally about 14 GB, which nearly fills the T4. It loads it in **4-bit** form, which shrinks it to about 4 GB. The model is a bit less precise, but it now fits with room left to train.

The tokenizer turns text into numbers the model can read. Mistral has no "padding" token, so the author reuses the end-of-text token for padding.

The author's code downloads the full 14.5 GB model and compresses it to 4-bit only after it arrives, which is slow on Colab. I load `unsloth/mistral-7b-instruct-v0.1-bnb-4bit` instead. It is the same Mistral-7B-Instruct-v0.1 model, already saved in 4-bit form, so the download is only about 4 GB.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "unsloth/mistral-7b-instruct-v0.1-bnb-4bit"  # same model, pre-saved in 4-bit

# Load the model with 4-bit quantization using BitsAndBytesConfig. 
# BitsAndBytesConfig allows to specify the quantization settings for loading the model in a memory-efficient manner.
# I am using 4-bit quantization to reduce the memory footprint of the model while maintaining reasonable performance.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# Load the model and tokenizer
# I am using AutoModelForCausalLM to load a causal language model, which is suitable for text generation tasks.
# The model is loaded with the specified quantization configuration and mapped to the appropriate device (GPU in this case).
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
)
model.config.use_cache = False

# Load the tokenizer for the model. The tokenizer is responsible for converting text into token IDs that the model can understand.
# I am setting the pad_token to be the same as the eos_token to ensure that padding is handled correctly during training and inference.
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

W1025 20:06:27.871000 3058 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
/Users/anishsamuel/dsc530/miniconda3/envs/book_env/lib/python3.13/site-packages/transformers/quantizers/auto.py:279: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

## Step 3: Prepare the data

The author uses 1,000 examples from the **Dolly** dataset. Each example has an instruction, sometimes some context, and a human-written answer.

Each example is turned into one block of text with headings: `### Instruction:`, `### Input:` and `### Response:`. This shows the model where the question ends and the answer starts.

*Two fixes:* the article's dataset name is wrong (the real one is `databricks/databricks-dolly-15k`), and the article never converts the text into tokens, which training needs. I added a short tokenize step.

In [ ]:
from datasets import load_dataset

# Load the Dolly dataset from Hugging Face's datasets library. 
# The dataset contains instruction-response pairs for training language models.
dataset = load_dataset("databricks/databricks-dolly-15k", split="train[:1000]")

# Define a function to format the prompt for the model.
# I am creating a structured prompt that includes the instruction, context and the expected response.
def format_prompt(sample):
    if sample["context"]:
        text = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
### Instruction:
{sample["instruction"]}

### Input:
{sample["context"]}

### Response:
{sample["response"]}"""
    else:
        text = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{sample["instruction"]}

### Response:
{sample["response"]}"""
    return {"text": text}

dataset = dataset.map(format_prompt)
print(dataset[0]["text"])

# I am defining a function to tokenize the dataset. The tokenizer converts the text into token IDs that the model can understand. 
# I am using truncation to ensure that the input does not exceed the maximum length of 512 tokens.
def tokenize(sample):
    return tokenizer(sample["text"], truncation=True, max_length=512)

tokenized_dataset = dataset.map(tokenize, remove_columns=dataset.column_names)

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
When did Virgin Australia start operating?

### Input:
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.

### Response:
Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.


## Step 4: Add the LoRA adapters

The author's settings:
- `r=16`: the adapter size. Bigger means it can learn more, but uses more memory.
- `lora_alpha=32`: how strongly the adapters affect the model's output.
- `target_modules=["q_proj", "v_proj"]`: put adapters only in two parts of the model's attention layers.
- `lora_dropout=0.05`: switches off a few connections at random during training so the model doesn't just memorize the examples.

I added `prepare_model_for_kbit_training`, a standard one-line step that saves memory so training fits on the T4.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# I am preparing the model for k-bit training, which is a technique to fine-tune large language models with reduced precision.
# This helps in saving memory and computational resources while still allowing effective training.
# The LoraConfig specifies the configuration for Low-Rank Adaptation (LoRA), which is a method to fine-tune large models efficiently by adding trainable low-rank matrices to certain layers of the model.
model = prepare_model_for_kbit_training(model)

# I am configuring the LoRA settings for fine-tuning the model. The parameters include:
# - r: The rank of the low-rank matrices. 16 is a common choice that balances performance and efficiency.
# - lora_alpha: The scaling factor for the LoRA layers. 32 is a typical value that helps in stabilizing training.
# - target_modules: The specific modules in the model where LoRA will be applied. q_proj and v_proj are common choices for transformer models, as they correspond to the query and value projections in the attention mechanism.
# - lora_dropout: The dropout rate for the LoRA layers to prevent overfitting. 0.05 is a standard choice that provides regularization without significantly impacting performance.
# - bias: Specifies how biases are handled in the LoRA layers. none means that no additional bias parameters are added, which can simplify the model and reduce the number of trainable parameters. 
#          The options for bias can be "none", "all", or "lora_only", depending on whether you want to include biases in the LoRA layers or not.
# - task_type: The type of task the model is being fine-tuned for, in this case, causal language modeling (CAUSAL_LM).
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# I am applying the LoRA configuration to the model to create a PEFT (Parameter-Efficient Fine-Tuning) model.
# The PEFT model allows for efficient fine-tuning of large language models by only training a small subset of parameters, which reduces memory usage and speeds up training.
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 6,815,744 || all params: 7,248,547,840 || trainable%: 0.0940


The output shows that only about **0.1%** of the model's parameters are being trained. That small number is why LoRA is fast and cheap.

## Step 5: Train

The author's settings:
- **Batch size 4, gradient accumulation 4:** the model looks at 4 examples at a time but only updates every 4 batches, so each update is based on 16 examples.
- **Learning rate 2e-4:** how big each learning step is. LoRA usually uses a higher rate than full fine-tuning.
- **3 epochs:** the model goes through all 1,000 examples three times.
- **fp16:** uses lighter 16-bit math to save memory and time.

The **loss** printed during training measures how wrong the model's guesses are. It should go down.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# I am setting up the training arguments for the fine-tuning process. The parameters include:
# - output_dir: The directory where the model checkpoints and training logs will be saved.
# - per_device_train_batch_size: The batch size for training on each device (GPU).
# - gradient_accumulation_steps: The number of steps to accumulate gradients before performing a backward pass. This effectively increases the batch size without requiring more GPU memory.
# - learning_rate: The initial learning rate for the optimizer.
# - num_train_epochs: The number of epochs to train the model.
# - logging_steps: The frequency (in steps) at which training logs are recorded.
# - fp16: Whether to use mixed precision training (float16) to speed up training and reduce memory usage.
# - save_total_limit: The maximum number of checkpoints to keep. Older checkpoints will be deleted to save disk space.
# - report_to: Specifies where to report training metrics. "none" means that no reporting will be done, which can be useful for local training without
training_args = TrainingArguments(
    output_dir="./mistral-7b-lora-dolly",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    fp16=True,
    save_total_limit=2,
    report_to="none",
)

# I am creating a Trainer instance to handle the training loop. 
# The Trainer class from Hugging Face's Transformers library simplifies the training process by managing the forward and backward passes, optimization, and logging.
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()

/Users/anishsamuel/dsc530/miniconda3/envs/book_env/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/anishsamuel/dsc530/miniconda3/envs/book_env/lib/python3.13/site-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/Users/anishsamuel/dsc530/miniconda3/envs/book_env/lib/python3.13/site-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. St

Step,Training Loss
10,1.752263
20,1.495917
30,1.544883
40,1.475249
50,1.470351
60,1.454792
70,1.452090
80,1.449957
90,1.358150
100,1.449629


/Users/anishsamuel/dsc530/miniconda3/envs/book_env/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=189, training_loss=1.429697526195062, metrics={'train_runtime': 20290.3935, 'train_samples_per_second': 0.148, 'train_steps_per_second': 0.009, 'total_flos': 4.715162618560512e+16, 'train_loss': 1.429697526195062, 'epoch': 3.0})

For every word (token) in the training answers, the model guesses what comes next, and the loss measures how surprised it was by the real next word.

- The model learned. The loss went down from 1.75 to about 1.36. That is about 22% lower, so the model’s guesses got better.
- It learned fastest at the start. The biggest drop was in the first 20 steps. The model quickly learned the question-and-answer format, and after that it improved slowly.
- The model may have started to memorize. The model saw the same 1,000 examples three times. The loss dropped a little each time it started over, around steps 90 and 140. This could mean it was memorizing the examples instead of really learning. Without a separate test set, we can’t be sure.
- Training was very slow in my macbook. It took 5 hours and 36 minutes because it ran on my computer instead of a GPU. On a Google Colab GPU, it took less than an hour.

## Step 6: Test the model

We give the fine-tuned model a new instruction it hasn't seen, using the same format as the training data, and let it write the response.

In [7]:
prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Explain the difference between supervised and unsupervised machine learning in simple terms.

### Response:
"""

peft_model.config.use_cache = True
inputs = tokenizer(prompt, return_tensors="pt").to(peft_model.device)
output = peft_model.generate(**inputs, max_new_tokens=150, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Explain the difference between supervised and unsupervised machine learning in simple terms.

### Response:
Supervised learning is when the model is trained on a labeled dataset. The model is given input/output pairs and learns to predict the output for a given input. Unsupervised learning is when the model is trained on an unlabeled dataset. The model is given input data and learns to identify patterns in the data. For example, clustering is a common unsupervised learning task where the model groups similar data points together. Another example is dimensionality reduction where the model reduces the number of input features while preserving the most important information. 

In summary, supervised learning is when the model is trained on labeled data and unsupervised learning is when the model is trained on unlabeled data. 



## Evaluation

The article shows a full, realistic pipeline. Shrinking the model to 4-bit and training only small adapters lets a 7-billion-parameter model be fine-tuned on a free Colab GPU. Only about 0.1% of the parameters were trained.

#### Limitations
- Mistral-7B-*Instruct* already follows instructions well, so it is hard to tell how much the fine-tuning helped. A base model would show a clearer difference.
- There is no test set, so we can't measure whether the model got better or just memorized the 1,000 examples.
- Because the padding token is the same as the end-of-text token, the model may not learn when to stop and can keep writing after its answer.

#### My results
- The model learned. The loss went down from 1.75 to about 1.36. That is about 22% lower, so the model’s guesses got better.
- It learned fastest at the start. The biggest drop was in the first 20 steps. The model quickly learned the question-and-answer format, and after that it improved slowly.
- The model may have started to memorize. The model saw the same 1,000 examples three times. The loss dropped a little each time it started over, around steps 90 and 140. This could mean it was memorizing the examples instead of really learning. Without a separate test set, we can’t be sure.
- Training was very slow in my macbook. It took 5 hours and 36 minutes because it ran on my computer instead of a GPU. On a Google Colab GPU, it took less than an hour.

#### Conclusion
LoRA makes fine-tuning big models affordable by training a tiny add-on instead of the whole model. The details still matter, though: the data format, the tokenizer setup and the choice of base model all affect the result.
